In [ ]:
# Zadanie 8 -- Early stopping eksperyment
"""
Porownaj trening z i bez early_stopping.

Wymagania:
Dataset: Breast Cancer
Architektura: (64, 32), solver='adam', max_iter=1000
Wariant 1: early_stopping=False
Wariant 2: early_stopping=True, validation_fraction=0.15
Porownaj: epoki, czas, accuracy

Oczekiwany wynik:
Tabela porownawcza i wykresy loss
"""

import time
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

configs = [
("no_early",False),
("early",True)
]

results = {}

for name,early in configs:
    clf = MLPClassifier(
        hidden_layer_sizes=(64,32),
        solver="adam",
        max_iter=1000,
        early_stopping=early,
        validation_fraction=0.15
    )
    start=time.time()
    clf.fit(X_train,y_train)
    end=time.time()
    acc=clf.score(X_test,y_test)
    results[name]={
        "epochs":len(clf.loss_curve_),
        "time":end-start,
        "acc":acc,
        "loss":clf.loss_curve_
    }

for k in results:
    print(k)
    print(results[k])
    print()

plt.figure()

plt.figure(figsize=(8,5))

for k in results:
    plt.plot(results[k]["loss"], label=k)

plt.title("Porównanie treningu: Early Stopping vs Brak Early Stopping")
plt.xlabel("Epoka (Iteration)")
plt.ylabel("Wartość funkcji straty (Loss)")
plt.legend(title="Konfiguracja modelu")
plt.grid(True)
plt.show()

In [ ]:
# Zadanie 16 - Porownanie 4 optymalizatorow od zera
"""
=== MODYFIKACJA Z LEKCJI - CO SIĘ DA ZAIMPORTOWAĆ Z SKLEARN

Zaimplementuj SGD, SGD+Momentum, RMSprop, Adam i porownaj na 3 datasetach.

Datasety:
make_moons, make_circles, make_classification

Wymagania:
Siec: 2 --> 16 --> 8 --> 1
3000 epok, batch_size=32 (losowe mini-batche)
Dla kazdego: loss_curve i granice decyzyjne Tabela: optymalizator x dataset x finalna accuracy
Heatmapa wynikow

Oczekiwany wynik:
12 wykresow (4 opt x 3 datasety) + heatmapa
"""

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

np.random.seed(0)

# Activation functions

def relu(x):
    return np.maximum(0,x)

def relu_deriv(x):
    return (x>0).astype(float)

def sigmoid(x):
    return 1/(1+np.exp(-x))

# Forward pass

def forward(X,W1,b1,W2,b2,W3,b3):
    z1=X@W1+b1
    a1=relu(z1)
    z2=a1@W2+b2
    a2=relu(z2)
    z3=a2@W3+b3
    a3=sigmoid(z3)
    cache=(z1,a1,z2,a2,z3,a3)
    return a3,cache

# Loss

def bce(y,p):
    eps=1e-8
    return -np.mean(y*np.log(p+eps)+(1-y)*np.log(1-p+eps))

# Backprop

def backward(X,y,W1,W2,W3,cache):
    z1,a1,z2,a2,z3,a3=cache
    m=len(X)
    dz3=a3-y
    dW3=a2.T@dz3/m
    db3=np.mean(dz3,axis=0)
    da2=dz3@W3.T
    dz2=da2*relu_deriv(z2)
    dW2=a1.T@dz2/m
    db2=np.mean(dz2,axis=0)
    da1=dz2@W2.T
    dz1=da1*relu_deriv(z1)
    dW1=X.T@dz1/m
    db1=np.mean(dz1,axis=0)
    return dW1,db1,dW2,db2,dW3,db3

# RMSprop optimizer (manual)

class RMSprop:

    def __init__(self,params,beta=0.9):
        self.s=[np.zeros_like(p) for p in params]
        self.beta=beta
        self.eps=1e-8

    def update(self,params,grads,lr):
        for i in range(len(params)):
            self.s[i]=self.beta*self.s[i]+(1-self.beta)*(grads[i]**2)
            params[i]-=lr*grads[i]/(np.sqrt(self.s[i])+self.eps)

# Train RMSprop network

def train_rmsprop(X,y):
    W1=np.random.randn(2,16)*0.1
    b1=np.zeros((1,16))
    W2=np.random.randn(16,8)*0.1
    b2=np.zeros((1,8))
    W3=np.random.randn(8,1)*0.1
    b3=np.zeros((1,1))
    params=[W1,b1,W2,b2,W3,b3]
    optimizer=RMSprop(params)
    lr=0.01
    batch_size=32
    epochs=3000

    for epoch in range(epochs):
        idx=np.random.permutation(len(X))
        for i in range(0,len(X),batch_size):
            batch=idx[i:i+batch_size]
            Xb=X[batch]
            yb=y[batch]
            out,cache=forward(Xb,W1,b1,W2,b2,W3,b3)
            grads=backward(Xb,yb,W1,W2,W3,cache)
            optimizer.update(params,grads,lr)
    return params

# Decision boundary

def plot_boundary(model,X,y,ax,title,scaler=None):
    x_min,x_max=X[:,0].min()-1,X[:,0].max()+1
    y_min,y_max=X[:,1].min()-1,X[:,1].max()+1
    xx,yy=np.meshgrid(
        np.linspace(x_min,x_max,200),
        np.linspace(y_min,y_max,200)
    )
    grid=np.c_[xx.ravel(),yy.ravel()]
    if scaler is not None:
        grid_scaled=scaler.transform(grid)
        preds=model.predict(grid_scaled)
    else:
        W1,b1,W2,b2,W3,b3=model
        probs,_=forward(grid,W1,b1,W2,b2,W3,b3)
        preds=(probs>0.5).astype(int)
    Z=preds.reshape(xx.shape)
    ax.contourf(xx,yy,Z,alpha=0.4,cmap="coolwarm")
    ax.scatter(X[:,0],X[:,1],c=y.flatten(),cmap="coolwarm",edgecolors="k")
    ax.set_title(title)

# Datasets

datasets={
"moons":make_moons(n_samples=600,noise=0.3),
"circles":make_circles(n_samples=600,noise=0.2,factor=0.5),
"classification":make_classification(
    n_samples=600,
    n_features=2,
    n_redundant=0,
    n_informative=2
)
}

optimizers=["SGD","Momentum","Adam","RMSprop"]

results=[]

fig,axes=plt.subplots(3,4,figsize=(16,12))

# Experiments

for r,(name,data) in enumerate(datasets.items()):
    X,y=data
    y=y.reshape(-1,1)
    X_train,X_test,y_train,y_test=train_test_split(
        X,y,test_size=0.3
    )
    scaler=StandardScaler()
    X_train=scaler.fit_transform(X_train)
    X_test=scaler.transform(X_test)

    for c,opt in enumerate(optimizers):
        if opt=="RMSprop":
            params=train_rmsprop(X_train,y_train)
            W1,b1,W2,b2,W3,b3=params
            pred,_=forward(X_test,W1,b1,W2,b2,W3,b3)
            acc=np.mean((pred>0.5)==y_test)
            plot_boundary(
                params,
                X,
                y,
                axes[r,c],
                f"{name}-{opt}\nacc={acc:.2f}"
            )
        else:
            if opt=="SGD":
                model=MLPClassifier(
                    hidden_layer_sizes=(16,8),
                    solver="sgd",
                    momentum=0,
                    max_iter=3000,
                    batch_size=32
                )
            elif opt=="Momentum":
                model=MLPClassifier(
                    hidden_layer_sizes=(16,8),
                    solver="sgd",
                    momentum=0.9,
                    max_iter=3000,
                    batch_size=32
                )
            else:
                model=MLPClassifier(
                    hidden_layer_sizes=(16,8),
                    solver="adam",
                    max_iter=3000,
                    batch_size=32
                )
            model.fit(X_train,y_train.ravel())
            acc=model.score(X_test,y_test)
            plot_boundary(
                model,
                X,
                y,
                axes[r,c],
                f"{name}-{opt}\nacc={acc:.2f}",
                scaler
            )
        results.append({
            "dataset":name,
            "optimizer":opt,
            "accuracy":acc
        })
plt.tight_layout()
plt.show()

# Results table

df=pd.DataFrame(results)
pivot=df.pivot(
    index="dataset",
    columns="optimizer",
    values="accuracy"
)
print(pivot)

# Heatmap

plt.figure(figsize=(6,4))
sns.heatmap(
    pivot,
    annot=True,
    cmap="viridis",
    fmt=".2f"
)
plt.title("Optimizer vs Dataset Accuracy")
plt.show()